# Voice Cloning Model

In [ ]:
#@title  CELL 2 — Clone Applio
import os
%cd /content
if not os.path.exists('/content/Applio'):
    !git clone https://github.com/IAHispano/Applio.git
else:
    print(' Applio already exists, skipping clone')
%cd /content/Applio
print('\n Applio ready!')

In [ ]:
#@title  CELL 3 — Install dependencies
%cd /content/Applio

!pip install -q \
    torch==2.7.1+cu128 \
    torchaudio==2.7.1+cu128 \
    torchvision==0.22.1+cu128 \
    --index-url https://download.pytorch.org/whl/cu128

import torch
print(f' Torch: {torch.__version__}')
print(f' CUDA: {torch.cuda.is_available()}')

!grep -Ev "^torch|^torchaudio|^torchvision" requirements.txt > requirements_fixed.txt
!pip install -q -r requirements_fixed.txt
!pip install -q torchcrepe torchfcpe pydub demucs

print('\n All installed! Ignore warnings above.')

In [ ]:
#@title  CELL 4 — Download pretrained models
%cd /content/Applio
!python core.py prerequisites --models true --exe true
print('\n Models downloaded!')

#CELL 5- Model config

In [4]:
import os, glob

MODEL_NAME = "myvoice"
APPLIO_DIR = "/content/Applio"
MODEL_DIR = f"{APPLIO_DIR}/logs/{MODEL_NAME}"

def _latest_pth(model_dir, model_name):
    files = sorted(glob.glob(f"{model_dir}/{model_name}*.pth"))
    return files[-1] if files else None

MODEL_PATH = _latest_pth(MODEL_DIR, MODEL_NAME)
INDEX_PATH = f"{MODEL_DIR}/{MODEL_NAME}.index"

# CELL 6 — Core functions

In [5]:
#  CELL 6 — Core functions
import os
import glob
import shutil
import subprocess

def _refresh_model_paths():
    global MODEL_PATH, INDEX_PATH
    MODEL_PATH = _latest_pth(MODEL_DIR, MODEL_NAME)
    INDEX_PATH = f"{MODEL_DIR}/{MODEL_NAME}.index"
    return MODEL_PATH, INDEX_PATH

def model_is_ready() -> bool:
    _refresh_model_paths()
    return bool(MODEL_PATH and os.path.exists(MODEL_PATH) and os.path.exists(INDEX_PATH))

def train_voice_model(
    dataset_path="/content/Applio/assets/datasets/myvoice",
    model_name=None,
    sample_rate=40000,
    total_epoch=5,
    batch_size=4,
    gpu="0",
):
    global MODEL_NAME, MODEL_DIR, INDEX_PATH

    if model_name:
        MODEL_NAME = model_name
    MODEL_DIR = f"{APPLIO_DIR}/logs/{MODEL_NAME}"
    INDEX_PATH = f"{MODEL_DIR}/{MODEL_NAME}.index"

    subprocess.run([
        "python", "core.py", "preprocess",
        "--model_name", MODEL_NAME,
        "--dataset_path", dataset_path,
        "--sample_rate", str(sample_rate),
        "--cut_preprocess", "Automatic",
    ], cwd=APPLIO_DIR, check=True)

    subprocess.run([
        "python", "core.py", "extract",
        "--model_name", MODEL_NAME,
        "--f0_method", "rmvpe",
        "--cpu_cores", "4",
        "--gpu", str(gpu),
        "--sample_rate", str(sample_rate),
        "--include_mutes", "2",
    ], cwd=APPLIO_DIR, check=True)

    subprocess.run([
        "python", "core.py", "train",
        "--model_name", MODEL_NAME,
        "--save_every_epoch", "10",
        "--save_only_latest", "False",
        "--save_every_weights", "True",
        "--total_epoch", str(total_epoch),
        "--sample_rate", str(sample_rate),
        "--batch_size", str(batch_size),
        "--gpu", str(gpu),
        "--pretrained", "True",
        "--custom_pretrained", "False",
        "--overtraining_detector", "False",
        "--cache_data_in_gpu", "False",
        "--index_algorithm", "Auto",
    ], cwd=APPLIO_DIR, check=True)

    # Detect both: myvoice.pth and myvoice_*.pth
    pths = sorted([
        f for f in os.listdir(MODEL_DIR)
        if f.startswith(MODEL_NAME) and f.endswith(".pth")
    ])
    if not pths:
        raise RuntimeError("Training finished but no .pth model found.")

    _refresh_model_paths()
    return {"model_path": MODEL_PATH, "index_path": INDEX_PATH, "ready": model_is_ready()}

def convert_vocals(
    input_vocals_path="/content/vocals.wav",
    output_path="/content/converted_vocals.wav",
    pitch=0,
    index_rate=0.5,
):
    if not model_is_ready():
        raise RuntimeError("Model not ready. Train or load model first.")

    subprocess.run([
        "python", "core.py", "infer",
        "--pitch", str(pitch),
        "--index_rate", str(index_rate),
        "--volume_envelope", "1.0",
        "--protect", "0.45",
        "--f0_method", "fcpe",
        "--input_path", input_vocals_path,
        "--output_path", output_path,
        "--pth_path", MODEL_PATH,
        "--index_path", INDEX_PATH,
        "--split_audio", "True",
        "--f0_autotune", "False",
        "--export_format", "WAV",
    ], cwd=APPLIO_DIR, check=True)

    return output_path


# # CELL 7 — FastAPI + ngrok

In [ ]:
# CELL 7 — FastAPI + ngrok

!pip install -q fastapi uvicorn python-multipart pyngrok nest_asyncio

from fastapi import FastAPI, UploadFile, File, Form
from fastapi.middleware.cors import CORSMiddleware
from fastapi.staticfiles import StaticFiles
from fastapi.responses import JSONResponse, FileResponse
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import shutil
import os
import asyncio

# Ngrok Authentication
ngrok.set_auth_token("3CoMnMyK5CfLDNODkxptunof3mD_3Ukd5AHNjSLqaTdJMyh4z")

# Folder creation
os.makedirs("/content/uploads", exist_ok=True)
os.makedirs("/content/outputs", exist_ok=True)

app = FastAPI()

# 1. CORS Configuration for Flutter Web
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
    expose_headers=["*"],
)

# Static files mounting
app.mount("/outputs", StaticFiles(directory="/content/outputs"), name="outputs")

PUBLIC_URL = None

def run_real_clone(voice_path, vocal_path, output_path, pitch=0):
    print(" Starting cloning...")

    try:
        # Prepare input files
        shutil.copy(voice_path, "/content/voice_input.wav")
        shutil.copy(vocal_path, "/content/vocals.wav")

        if model_is_ready():
            print(" Using existing model")
        else:
            print(" Training model...")
            dataset_dir = "/content/Applio/assets/datasets/myvoice"
            os.makedirs(dataset_dir, exist_ok=True)
            shutil.copy("/content/voice_input.wav", f"{dataset_dir}/voice.wav")
            train_voice_model(dataset_path=dataset_dir)

        convert_vocals(
            input_vocals_path="/content/vocals.wav",
            output_path="/content/converted_vocals.wav",
            pitch=pitch
        )

        # Save final output
        shutil.copy("/content/converted_vocals.wav", output_path)
        print("  Done:", output_path)
    except Exception as e:
        print(f" Error in model logic: {e}")
        raise e

@app.post("/clone/")
async def clone_voice(
    voice: UploadFile = File(...),
    song: UploadFile = File(...),
    pitch: int = Form(0),
):
    try:
        global PUBLIC_URL
        # File names handle කිරීම
        voice_path = f"/content/uploads/{voice.filename}"
        song_path = f"/content/uploads/{song.filename}"

        # Put stable name to Output file
        output_filename = "final_song.wav"
        output_path = f"/content/outputs/{output_filename}"

        # Uploads save
        with open(voice_path, "wb") as f:
            shutil.copyfileobj(voice.file, f)
        with open(song_path, "wb") as f:
            shutil.copyfileobj(song.file, f)

        # Run model logic
        run_real_clone(voice_path, song_path, output_path, pitch)

        # 2. FIX: Constructing the DIRECT static URL

        final_url = f"{PUBLIC_URL}/outputs/{output_filename}?ngrok-skip-browser-warning=true"

        return {
           "status": "success",
           "outputUrl": final_url
        }

    except Exception as e:
        return JSONResponse({"error": str(e)}, status_code=500)

# Start Ngrok and Uvicorn
nest_asyncio.apply()
ngrok.kill()

public_url = ngrok.connect(8000)
PUBLIC_URL = public_url.public_url

print(" COLAB API URL:", PUBLIC_URL)
print(" CLONE ENDPOINT:", PUBLIC_URL + "/clone/")

config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)

await server.serve()